In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split, ParameterGrid
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support
import json
import time

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
root_dir = r'smaller_dataset.csv'
df = pd.read_csv(root_dir)

In [ ]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Uploading_Attack",
    "DoS_DoS SYN Flood",
    "DDoS_DDoS ACK Fragmentation",
    "DDoS_DDoS-HTTP Flood"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  # replace with your actual column names

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [ ]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [ ]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':
        return 0
    else:
        return 1

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

In [ ]:
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]

inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
df = df.dropna()
df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
labels = df['Anomaly_Label']
attack_types = df['Attack_Type']  #to track which attacks are detected

feature_combos = [
   ['SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Flow IAT Min'],
    ['SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'SYN Flag Count', 'Flow IAT Min'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Dst Port_freq', 'Flow IAT Min', 'Flow IAT Mean'],
    ['Fwd Segment Size Avg', 'SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Flow IAT Min', 'Flow IAT Mean'],
    ['SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Flow IAT Min'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Dst Port_freq', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Fwd Packet Length Max', 'SYN Flag Count', 'Idle Mean', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'SYN Flag Count', 'Dst Port_freq', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Dst Port_freq', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
  ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Fwd Packet Length Max', 'SYN Flag Count', 'Dst Port_freq', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Flow IAT Max', 'Dst Port_freq', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
    ['FWD Init Win Bytes', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Flow IAT Max', 'Idle Mean', 'Average Packet Size', 'Flow IAT Min', 'Flow IAT Mean'],
['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Packet Length Max', 'Dst Port_freq', 'Idle Mean', 'Packet Length Mean', 'Flow IAT Min'],
  ['FWD Init Win Bytes', 'Fwd Segment Size Avg', 'Src IP_freq', 'SYN Flag Count', 'Total Length of Fwd Packet', 'Dst Port_freq', 'Average Packet Size', 'Packet Length Mean', 'Flow IAT Min', 'Flow IAT Mean']
]

In [ ]:
param_grid = {
    'n_estimators': [ 100, 200, 300],
    'max_samples': [0.5,0.7, 0.8],
    'contamination': [0.01, 0.05,0.1],
    'max_features': [0.7, 0.8]
}

In [ ]:
import json
from collections import defaultdict

output_file = "IForestHPWithoutDoS.jsonl"

scaler = StandardScaler()

with open(output_file, 'w', encoding='utf-8') as f_out:
    for features in feature_combos:

        X = df[features]
       
        X_scaled = scaler.fit_transform(X)

        X_train, X_test, Y_train, Y_test, attacks_train, attacks_test = train_test_split(
            X, labels, attack_types,
            test_size=0.3,
            random_state=42
        )


        for params in ParameterGrid(param_grid):
            model = IsolationForest(**params, random_state=42)
            model.fit(X_train)

            preds = model.predict(X_test)
            preds = np.where(preds == -1, 1, 0)

            score = f1_score(Y_test, preds)

            TP = np.sum((Y_test == 1) & (preds == 1))
            FN = np.sum((Y_test == 1) & (preds == 0))
            TN = np.sum((Y_test == 0) & (preds == 0))
            FP = np.sum((Y_test == 0) & (preds == 1))

            total_attacks = TP + FN
            total_normals = TN + FP

            # Percentages
            attack_detected_pct = (TP / total_attacks * 100) if total_attacks else 0
            attack_missed_pct = (FN / total_attacks * 100) if total_attacks else 0
            normal_correct_pct = (TN / total_normals * 100) if total_normals else 0
            normal_wrong_pct = (FP / total_normals * 100) if total_normals else 0


            attack_report = defaultdict(dict)
            true_anomalies = (Y_test == 1)

            for attack in np.unique(attacks_test[true_anomalies]):
                attack_mask = (attacks_test == attack)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    Y_test[attack_mask], preds[attack_mask], pos_label=1, average='binary', zero_division=0
                )
                attack_report[attack] = {
                    'precision': round(precision, 3),
                    'recall': round(recall, 3),
                    'f1': round(f1, 3)
                }

            detected_attacks = {atk for atk, vals in attack_report.items() if vals['recall'] > 0}
            missed_attacks = {atk for atk in np.unique(attacks_test[true_anomalies]) if atk not in detected_attacks}

            result = {
            'features': features,
            'params': params,
            'f1_score': round(score, 4),
            'per_attack_report': dict(attack_report),
            'detected_attacks': list(detected_attacks),
            'missed_attacks': list(missed_attacks),
            'Attacks correctly detected': f"{TP} / {total_attacks} ({attack_detected_pct:.3f}%)",
            'Attacks missed': f"{FN} / {total_attacks} ({attack_missed_pct:.3f}%)",
            'Normal instances correctly identified': f"{TN} / {total_normals} ({normal_correct_pct:.3f}%)",
            'Normal instances misclassified as attack': f"{FP} / {total_normals} ({normal_wrong_pct:.3f}%)"
            }


           
            f_out.write(json.dumps(result) + "\n")


            print("\n--- Iteration Result ---")
            print(f"F1 Score: {score:.4f}")
            print(f"Features: {features}")
            print(f"Params: {params}")

            print("\nPer-Attack Report:")
            for atk, vals in attack_report.items():
                print(f"  {atk}: Precision={vals['precision']}, Recall={vals['recall']}, F1={vals['f1']}")

            print(f"Detected attacks: {detected_attacks}")
            print(f"Missed attacks: {missed_attacks}")



print("Results save to file.")



In [ ]:
import json

output_file = "IForestHPWithoutDoS.jsonl"
results_path = output_file

target_feature_count = 7 

with open(results_path, "r") as f:
    results = [json.loads(line) for line in f]

sorted_results = sorted(results, key=lambda x: x['f1_score'], reverse=True)

top_n = 10 

for i, result in enumerate(sorted_results[:top_n], 1):
    print(f"\n--- Rank #{i} ---")
    print(f"F1 Score: {result['f1_score']}")
    print(f"Features ({len(result['features'])}): {result['features']}")
    print(f"Params: {result['params']}")
    print("Detected Attacks:", result["detected_attacks"])
    print("Missed Attacks:", result["missed_attacks"])
    print("Per-Attack F1s:")
    for atk, vals in result["per_attack_report"].items():
        print(f"  {atk}: F1={vals['f1']}, Precision={vals['precision']}, Recall={vals['recall']}")
